# Phase 1 · Reward benchmark (Kaggle or Colab, one T4 GPU)

Scores English→German WMT translations that experts rated (MQM) with each candidate reward, measures how well each reward agrees with the experts **when comparing translations of the same source**, probes each reward's blind spots, and suggests the training reward (R_train) and held-out evaluator (R_eval).
Expected time: **about 1.5–2 hours on one T4**. Details: [`docs/workflow.md` §4](https://github.com/guohere/machine-translation-with-RLHF/blob/claude/compassionate-ritchie-2nga86/docs/workflow.md).

### One-time setup
1. **Hugging Face:** open [Unbabel/wmt22-cometkiwi-da](https://huggingface.co/Unbabel/wmt22-cometkiwi-da) and accept the license (gated, non-commercial use). Create a *read* token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
2. **GitHub (recommended):** create a fine-grained token at [github.com/settings/personal-access-tokens](https://github.com/settings/personal-access-tokens) → *Generate new token*: repository access **Only select repositories → machine-translation-with-RLHF**, permission **Contents: Read and write**, expiration 30 days. With it, the notebook saves the scores to the repo (`results/p1/`) after each reward, so nothing is lost if the session stops, and a re-run continues from what was saved.
3. **Kaggle:** *Settings* → Accelerator **GPU T4** · Internet **On** (needs a phone-verified account). *Add-ons → Secrets* → add `HF_TOKEN` (and `GH_TOKEN`), tick both for this notebook.
   **Colab:** *Runtime → Change runtime type* → **T4 GPU**. 🔑 *Secrets* panel → add `HF_TOKEN` (and `GH_TOKEN`), enable notebook access.
4. **Run all** and keep the tab open, or on Kaggle use **Save Version → Save & Run All (Commit)** to run in the background.

### When it finishes
With `GH_TOKEN`, the results are already in the repo under `results/p1/`. Without it, **download `p1_results.zip` before stopping the session**: files of an interactive session are deleted when it stops.

In [ ]:
# ---- configuration ----
REPO = "https://github.com/guohere/machine-translation-with-RLHF.git"
BRANCH = "claude/compassionate-ritchie-2nga86"
TESTSETS = "wmt22 wmt23.sent"   # WMT en-de test sets with expert MQM scores
REWARDS = "chrf bleu cometkiwi22 comet22 metricx24_qe metricx24_ref"
RUN = "runs/p1"

In [ ]:
# ---- GPU check and Hugging Face token (works on Kaggle and Colab) ----
import os, subprocess

WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
os.chdir(WORK)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the accelerator to a T4 GPU before continuing")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded")
else:
    print("WARNING: no HF_TOKEN secret found - the gated CometKiwi model will fail to download")

In [ ]:
# ---- get the code and install (about 2 minutes) ----
if not os.path.isdir("machine-translation-with-RLHF"):
    !git clone -q -b {BRANCH} {REPO}
else:
    !git -C machine-translation-with-RLHF pull -q
os.chdir(os.path.join(WORK, "machine-translation-with-RLHF"))
!git log -1 --format="code version: %h %s"
# Note: this pins transformers<5 for this session (COMET 2.2.7 needs the transformers 4 output layout).
!pip install -q -e ".[phase1]"
# Installed without their declared dependencies on purpose (see pyproject.toml):
# unbabel-comet pins numpy<2 / transformers<5, mt-metrics-eval pulls apache_beam.
!pip install -q --no-deps unbabel-comet==2.2.7 "git+https://github.com/google-research/mt-metrics-eval"
!python -c "import torch, transformers, comet, mt_metrics_eval, mtrlhf; print('ready: torch', torch.__version__, '| transformers', transformers.__version__)"

In [ ]:
# ---- save results to GitHub as the run goes (optional, needs the GH_TOKEN secret) ----
import base64, glob, shutil, subprocess

GH_TOKEN = get_secret("GH_TOKEN")
RESULTS = "results/p1"
print("GH_TOKEN loaded: results will be saved to GitHub as they finish" if GH_TOKEN else
      "No GH_TOKEN secret: results stay in this session only - download p1_results.zip before stopping")

def _git(*args):
    auth = base64.b64encode(f"x-access-token:{GH_TOKEN}".encode()).decode()
    cmd = ["git", "-c", "user.name=Phase 1 notebook", "-c", "user.email=noreply@users.noreply.github.com",
           "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}", *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).replace(GH_TOKEN, "***").replace(auth, "***")

def save_results(message):
    """Copy scores and report into results/p1/ and push them to the branch."""
    os.makedirs(RESULTS, exist_ok=True)
    for sub in ("scores", "report"):
        if os.path.isdir(f"{RUN}/{sub}"):
            shutil.copytree(f"{RUN}/{sub}", f"{RESULTS}/{sub}", dirs_exist_ok=True)
    if os.path.exists(f"{RUN}/inputs_meta.json"):
        shutil.copy(f"{RUN}/inputs_meta.json", RESULTS)
    if not GH_TOKEN:
        return
    _git("add", RESULTS)
    if _git("diff", "--cached", "--quiet")[0] != 0:
        code, out = _git("commit", "-q", "-m", message)
        if code:
            print("commit failed:", out); return
    code, out = _git("pull", "-q", "--rebase", "origin", BRANCH)
    if code:
        print("pull failed:", out); return
    code, out = _git("push", "-q", "origin", f"HEAD:{BRANCH}")
    print(f"saved to GitHub: {message}" if code == 0 else f"push failed: {out}")

def restore_results():
    """Reuse scores saved by an earlier run, so scoring continues instead of starting over."""
    n = 0
    for f in glob.glob(f"{RESULTS}/scores/*"):
        dst = os.path.join(RUN, "scores", os.path.basename(f))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(f, dst)
            n += 1
    print(f"restored {n} score files saved by an earlier run" if n else "no earlier scores found: starting fresh")

### Step 1 · Download the WMT data and build the inputs (CPU, ~5–10 min)
Streams Google's mt-metrics-eval archive, keeps only the needed test sets, and writes `segments.jsonl` (rated translations) and `probes.jsonl` (blind-spot probes).

In [ ]:
!python scripts/p1_build_inputs.py --run {RUN} --testsets {TESTSETS} --download

### Step 2 · Smoke test (~3 min)
Scores 16 rows with the two neural QE rewards, to catch login or memory problems before the long run.

In [ ]:
!python scripts/p1_score.py --run {RUN} --rewards cometkiwi22 metricx24_qe --limit 16

### Step 3 · Score everything (~1–1.5 h on a T4)
One reward at a time; after each one the scores are saved to GitHub (with `GH_TOKEN`). If the session stops, run the notebook again: saved rewards are skipped and an unfinished one continues where it stopped.

In [ ]:
restore_results()
for reward in REWARDS.split():
    !python scripts/p1_score.py --run {RUN} --rewards {reward}
    save_results(f"Phase 1: scores for {reward}")

### Step 4 · Analyze (CPU, ~2 min)

In [ ]:
!python scripts/p1_analyze.py --run {RUN} > /dev/null
save_results("Phase 1: report")
from IPython.display import Markdown, Image, display
summary = open(f"{RUN}/report/summary.md").read()
display(Markdown(summary.replace("![blind spots](blind_spots.png)", "")))
display(Image(f"{RUN}/report/blind_spots.png"))

In [ ]:
# ---- the same summary as plain text, for copying into a chat ----
print(summary)

In [ ]:
# ---- package the results for download (needed only without GH_TOKEN) ----
pkg = os.path.join(WORK, "p1_results")
shutil.rmtree(pkg, ignore_errors=True)
shutil.copytree(f"{RUN}/report", f"{pkg}/report")
shutil.copytree(f"{RUN}/scores", f"{pkg}/scores")
shutil.copy(f"{RUN}/inputs_meta.json", pkg)
print("download:", shutil.make_archive(pkg, "zip", pkg),
      "(Kaggle: right panel Output -> /kaggle/working; Colab: Files panel)")

### Troubleshooting
| Symptom | Fix |
|---|---|
| `Permission 'kernelSessions.enableInternet' was denied` | Verify your phone number: Kaggle profile picture → *Settings* → *Phone verification*. Then *Settings → Internet → On* |
| `401` / `403` / "gated repo" when loading `wmt22-cometkiwi-da` | Accept the license on the model page, and check the `HF_TOKEN` secret is attached to this notebook |
| `push failed` | Check the `GH_TOKEN` secret: fine-grained token for this repository with *Contents: Read and write*, not expired |
| "Temporary failure in name resolution" / downloads hang | Turn **Internet On** |
| `CUDA out of memory` while scoring MetricX | Add `--max-tokens-per-batch 4096` to the command in step 3 |
| The session stopped during step 3 | Run the notebook again: rewards saved to GitHub are skipped |
| *Save Version* failed | Open the failed version (*Version* count next to *Save Version*) → *Logs*, and look at the last error |